In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [2]:

!wget -O github_input.txt "https://github.com/yoosufaamir/HPC-.git"

--2026-10-08 18:45:13--  https://github.com/yoosufaamir/HPC-.git
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/yoosufaamir/HPC- [following]
--2026-10-08 18:45:14--  https://github.com/yoosufaamir/HPC-
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘github_input.txt’

github_input.txt        [  <=>               ] 231.46K   977KB/s    in 0.2s    

2026-10-08 18:45:15 (977 KB/s) - ‘github_input.txt’ saved [237011]



In [5]:
!./word_count github_input.txt 4

/bin/bash: line 1: ./word_count: No such file or directory


In [6]:
!git clone https://github.com/yoosufaamir/HPC-.git

Cloning into 'HPC-'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 698.61 KiB | 2.89 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [7]:
%%writefile word_count.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <pthread.h>

#define HASH_SIZE 10007
#define MAX_WORD_LEN 64
#define OUTPUT_FILE "result.txt"

typedef struct WordNode {
    char word[MAX_WORD_LEN];
    long count;
    struct WordNode *next;
} WordNode;

WordNode *hashTable[HASH_SIZE];
pthread_mutex_t tableLock;

unsigned int hashWord(const char *word) {
    unsigned long hash = 5381;
    int c;
    while ((c = *word++)) {
        hash = ((hash << 5) + hash) + c;
    }
    return (unsigned int)(hash % HASH_SIZE);
}

void addWordToTable(const char *word) {
    unsigned int idx = hashWord(word);
    pthread_mutex_lock(&tableLock);
    WordNode *node = hashTable[idx];
    while (node != NULL) {
        if (strcmp(node->word, word) == 0) {
            node->count++;
            pthread_mutex_unlock(&tableLock);
            return;
        }
        node = node->next;
    }
    WordNode *newNode = malloc(sizeof(WordNode));
    strncpy(newNode->word, word, MAX_WORD_LEN - 1);
    newNode->word[MAX_WORD_LEN - 1] = '\0';
    newNode->count = 1;
    newNode->next = hashTable[idx];
    hashTable[idx] = newNode;
    pthread_mutex_unlock(&tableLock);
}

struct ThreadArgs {
    long start;
    long end;
    char **words;
};

void *countWords(void *args) {
    struct ThreadArgs *targs = (struct ThreadArgs *)args;
    pthread_t tid = pthread_self();
    long processed = 0;
    for (long i = targs->start; i <= targs->end; i++) {
        addWordToTable(targs->words[i]);
        processed++;
    }
    printf("Thread %lu processed %ld words (range %ld - %ld)\n",
           (unsigned long)tid, processed, targs->start, targs->end);
    pthread_exit(NULL);
}

void cleanWord(char *word) {
    int i, j = 0;
    char buffer[MAX_WORD_LEN];
    for (i = 0; word[i] != '\0' && j < MAX_WORD_LEN - 1; i++) {
        if (isalnum((unsigned char)word[i])) {
            buffer[j++] = (char)tolower((unsigned char)word[i]);
        }
    }
    buffer[j] = '\0';
    strcpy(word, buffer);
}

typedef struct {
    char word[MAX_WORD_LEN];
    long count;
} ResultEntry;

int compareEntries(const void *a, const void *b) {
    const ResultEntry *ea = (const ResultEntry *)a;
    const ResultEntry *eb = (const ResultEntry *)b;
    if (eb->count != ea->count) {
        return (int)(eb->count - ea->count);
    }
    return strcmp(ea->word, eb->word);
}

int main(int argc, char *argv[]) {
    if (argc != 3) {
        printf("Usage: %s <inputFile> <threadCount>\n", argv[0]);
        return 1;
    }
    char *fileName = argv[1];
    int threadCount = atoi(argv[2]);
    if (threadCount <= 0) {
        printf("threadCount must be a positive integer.\n");
        return 1;
    }
    FILE *fp = fopen(fileName, "r");
    if (fp == NULL) {
        perror("Error opening input file");
        return 1;
    }
    long capacity = 1024;
    long wordCount = 0;
    char **words = malloc(capacity * sizeof(char *));
    char buffer[MAX_WORD_LEN];
    while (fscanf(fp, "%63s", buffer) == 1) {
        cleanWord(buffer);
        if (strlen(buffer) == 0) {
            continue;
        }
        if (wordCount >= capacity) {
            capacity *= 2;
            words = realloc(words, capacity * sizeof(char *));
        }
        words[wordCount] = malloc(strlen(buffer) + 1);
        strcpy(words[wordCount], buffer);
        wordCount++;
    }
    fclose(fp);
    printf("Read %ld words from %s\n", wordCount, fileName);
    if (wordCount == 0) {
        printf("No words found in file.\n");
        free(words);
        return 1;
    }
    if (threadCount > wordCount) {
        threadCount = (int)wordCount;
    }
    long sliceList[threadCount];
    long remainder = wordCount % threadCount;
    for (int i = 0; i < threadCount; i++) {
        sliceList[i] = wordCount / threadCount;
    }
    for (int j = 0; j < remainder; j++) {
        sliceList[j] = sliceList[j] + 1;
    }
    long startList[threadCount];
    long finishList[threadCount];
    for (int k = 0; k < threadCount; k++) {
        if (k == 0) {
            startList[k] = 0;
            finishList[k] = startList[k] + sliceList[k] - 1;
        } else {
            startList[k] = finishList[k - 1] + 1;
            finishList[k] = startList[k] + sliceList[k] - 1;
        }
    }
    pthread_mutex_init(&tableLock, NULL);
    pthread_t tids[threadCount];
    struct ThreadArgs targs[threadCount];
    for (int i = 0; i < threadCount; i++) {
        targs[i].start = startList[i];
        targs[i].end = finishList[i];
        targs[i].words = words;
        pthread_create(&tids[i], NULL, countWords, &targs[i]);
    }
    for (int i = 0; i < threadCount; i++) {
        pthread_join(tids[i], NULL);
    }
    pthread_mutex_destroy(&tableLock);
    long uniqueWords = 0;
    for (int i = 0; i < HASH_SIZE; i++) {
        WordNode *node = hashTable[i];
        while (node != NULL) {
            uniqueWords++;
            node = node->next;
        }
    }
    ResultEntry *results = malloc(uniqueWords * sizeof(ResultEntry));
    long r = 0;
    for (int i = 0; i < HASH_SIZE; i++) {
        WordNode *node = hashTable[i];
        while (node != NULL) {
            strcpy(results[r].word, node->word);
            results[r].count = node->count;
            r++;
            node = node->next;
        }
    }
    qsort(results, uniqueWords, sizeof(ResultEntry), compareEntries);
    FILE *out = fopen(OUTPUT_FILE, "w");
    if (out == NULL) {
        perror("Error opening output file");
        return 1;
    }
    fprintf(out, "%-30s %s\n", "WORD", "FREQUENCY");
    for (long i = 0; i < uniqueWords; i++) {
        fprintf(out, "%-30s %ld\n", results[i].word, results[i].count);
    }
    fclose(out);
    printf("Found %ld unique words. Results written to %s\n", uniqueWords, OUTPUT_FILE);
    for (long i = 0; i < wordCount; i++) {
        free(words[i]);
    }
    free(words);
    free(results);
    for (int i = 0; i < HASH_SIZE; i++) {
        WordNode *node = hashTable[i];
        while (node != NULL) {
            WordNode *temp = node;
            node = node->next;
            free(temp);
        }
    }
    return 0;
}

Writing word_count.c


In [8]:
!ls HPC-/

images	MatData.txt  passwords.txt  README.md  WordOccurrenceDataset.txt


In [11]:
!gcc -O3 -pthread word_count.c -o word_count
!./word_count HPC-/WordOccurrenceDataset.txt 4

Read 120000 words from HPC-/WordOccurrenceDataset.txt
Thread 133056452544192 processed 30000 words (range 90000 - 119999)
Thread 133056460936896 processed 30000 words (range 30000 - 59999)
Thread 133056341997248 processed 30000 words (range 60000 - 89999)
Thread 133056469329600 processed 30000 words (range 0 - 29999)
Found 94 unique words. Results written to result.txt
